# Multi-Scenario Comparison Dashboard

Compares two algorithm scenarios (`setup_1`) against the reconstructed production baseline.
S0 (`setup_0/output_50000`) uses different input parameters and is **not** included here.

| Scenario | Baseline | Algorithm | Key change |
|---|---|---|---|
| **S1-Speed** | `file_snapshots/output_payload.json` | `setup_1/output_speed` | `tiempo_previo=0`, OSRM distance + speed-based times |
| **S2-Time** | same | `setup_1/output_time` | `tiempo_previo=0`, OSRM duration times |

**Sections**
1. **KPI Overview** — both scenarios side-by-side, Δ vs production baseline
2. **S1 — OSRM Speed** — baseline ↔ algorithm detail
3. **S2 — OSRM Time** — baseline ↔ algorithm detail
4. **OSRM Speed vs Time** — direct method comparison including travel-time distribution

All plotting logic lives in `src/analysis/compare_solutions.py`. Edit the **Configuration** cell only.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
from typing import Optional

import pandas as pd
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_PROJECT_ROOT = Path("..").resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from src.analysis.compare_solutions import (
    load_and_prepare,
    build_overview_table,
    build_comparison_gantt_figure,
    build_comparison_service_figure,
    build_comparison_driver_figure,
)
from src.analysis.solution_evaluation import compute_payload_summary
from src.optimization.settings.model_params import ModelParams
from src.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────────────

BASELINE = Path("../experiments/phase2/cases/file_snapshots/output_payload.json")
S1_SPEED = Path("../experiments/phase2/results/setup_1/output_speed/output_payload.json")
S1_TIME  = Path("../experiments/phase2/results/setup_1/output_time/output_payload.json")

INPUT_FILE:       Optional[Path] = Path("../experiments/phase2/cases/input_mirror.json")
DRIVER_DIRECTORY: Optional[Path] = Path("../data/model_input/driver_directory.json")

PLANNING_DATE:   Optional[str] = "2026-02-18"
DISTANCE_METHOD: str           = DEFAULT_DISTANCE_METHOD

_params = ModelParams()
ALFRED_SPEED_KMH: float = _params.alfred_speed_kmh

print(f"distance_method={DISTANCE_METHOD!r}  |  alfred_speed_kmh={ALFRED_SPEED_KMH}")

distance_method='osrm'  |  alfred_speed_kmh=20.0


In [3]:
# ── Load SolutionPairs ─────────────────────────────────────────────────────────────────

_common = dict(
    input_file=INPUT_FILE,
    driver_directory=DRIVER_DIRECTORY,
    planning_date=PLANNING_DATE,
    distance_method=DISTANCE_METHOD,
    speed_kmh=ALFRED_SPEED_KMH,
)

print("── S1-Speed: baseline vs OSRM-speed algorithm ───────────────────────────")
pair_s1_speed = load_and_prepare(BASELINE, S1_SPEED, **_common)

print("\n── S2-Time: baseline vs OSRM-time algorithm ─────────────────────────────")
pair_s1_time = load_and_prepare(BASELINE, S1_TIME, **_common)

print("\n── Methods comparison: OSRM-speed vs OSRM-time ──────────────────────────")
pair_methods = load_and_prepare(S1_SPEED, S1_TIME, **_common)

── S1-Speed: baseline vs OSRM-speed algorithm ───────────────────────────
[driver_home_lookup] 13 drivers loaded
[coord_lookup] 57 labors | method='osrm'
[filter_by_date] Dropped 4 labor(s) before 2026-02-18.
sol_a: 53 labors | 12 drivers | 99 segments
sol_b: 44 labors | 12 drivers | 98 segments

── S2-Time: baseline vs OSRM-time algorithm ─────────────────────────────
[driver_home_lookup] 13 drivers loaded
[coord_lookup] 57 labors | method='osrm'
[filter_by_date] Dropped 4 labor(s) before 2026-02-18.
sol_a: 53 labors | 12 drivers | 99 segments
sol_b: 44 labors | 12 drivers | 100 segments

── Methods comparison: OSRM-speed vs OSRM-time ──────────────────────────
[driver_home_lookup] 13 drivers loaded
[coord_lookup] 57 labors | method='osrm'
sol_a: 44 labors | 12 drivers | 98 segments
sol_b: 44 labors | 12 drivers | 100 segments


---
## 1. KPI Overview — Both Scenarios

Both algorithm solutions compared side-by-side. **Δ columns** show delta vs the production baseline  
(green = improvement, red = degradation).

In [4]:
_sums = {
    "Baseline":  compute_payload_summary(pair_s1_speed.rows_a, segments=pair_s1_speed.segments_a),
    "S1-Speed":  compute_payload_summary(pair_s1_speed.rows_b, segments=pair_s1_speed.segments_b),
    "S2-Time":   compute_payload_summary(pair_s1_time.rows_b,  segments=pair_s1_time.segments_b),
}
_base = _sums["Baseline"]

_METRICS = [
    ("services_count",               "Services"),
    ("labors_count",                 "Labors"),
    ("labors_vt_count",              "Labors (VT)"),
    ("drivers_count",                "Drivers used"),
    ("labors_assigned",              "Labors assigned"),
    ("labors_infeasible",            "Labors infeasible"),
    ("labors_infeasible_pct",        "Infeasible (%)"),
    ("total_labor_distance_km",      "Labor dist (km)"),
    ("total_driver_move_distance_km","Move dist (km)"),
    ("total_distance_km",            "Total dist (km)"),
    ("avg_labor_distance_km",        "Avg labor dist (km)"),
    ("avg_driver_move_distance_km",  "Avg move dist (km)"),
]
_LOWER_IS_BETTER = {
    "Drivers used", "Labors infeasible", "Infeasible (%)",
    "Move dist (km)", "Total dist (km)", "Avg move dist (km)",
}
_HIGHER_IS_BETTER = {"Labors assigned"}

def _d(b, a):
    return round(a - b, 2) if (a is not None and b is not None) else None

_rows = []
for _key, _label in _METRICS:
    _b  = _base.get(_key)
    _s1 = _sums["S1-Speed"].get(_key)
    _s2 = _sums["S2-Time"].get(_key)
    _rows.append({
        "Metric":      _label,
        "Baseline":    _b,
        "S1-Speed":    _s1,
        "Δ S1-Speed":  _d(_b, _s1),
        "S2-Time":     _s2,
        "Δ S2-Time":   _d(_b, _s2),
    })

_df_overview = pd.DataFrame(_rows).set_index("Metric")
_DELTA_COLS  = ["Δ S1-Speed", "Δ S2-Time"]

def _color_deltas(df):
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for _m in df.index:
        for _c in _DELTA_COLS:
            if _c not in df.columns:
                continue
            _v = df.loc[_m, _c]
            if pd.isna(_v) or _v == 0:
                continue
            if _m in _LOWER_IS_BETTER:
                _ok = _v < 0
            elif _m in _HIGHER_IS_BETTER:
                _ok = _v > 0
            else:
                continue
            styles.loc[_m, _c] = (
                "background-color: #d4edda; color: #155724" if _ok
                else "background-color: #f8d7da; color: #721c24"
            )
    return styles

display(
    _df_overview.style
    .format(precision=2, na_rep="—")
    .apply(_color_deltas, axis=None)
    .set_caption("KPI Overview — Δ = Algorithm vs Production Baseline")
)

,Baseline,S1-Speed,Δ S1-Speed,S2-Time,Δ S2-Time
Metric,,,,,
Services,39.00,35.00,-4.00,35.00,-4.00
Labors,53.00,44.00,-9.00,44.00,-9.00
Labors (VT),45.00,39.00,-6.00,39.00,-6.00
Drivers used,12.00,12.00,0.00,12.00,0.00
Labors assigned,39.00,39.00,0.00,39.00,0.00
Labors infeasible,2.00,0.00,-2.00,0.00,-2.00
Infeasible (%),3.77,0.00,-3.77,0.00,-3.77
Labor dist (km),414.01,414.01,0.00,414.01,0.00
Move dist (km),427.87,403.72,-24.15,393.22,-34.65


---
## 2. S1 — OSRM Speed Method
Production baseline vs `setup_1/output_speed` (`tiempo_previo=0`, OSRM distance + speed-based travel times).

In [8]:
build_overview_table(pair_s1_speed, "baseline", "S1 — OSRM Speed")

,baseline,S1 — OSRM Speed,delta,delta_pct
metric,,,,
services,39.00,35.00,-4.00,-10.26
labors_total,53.00,44.00,-9.00,-16.98
labors_vt,45.00,39.00,-6.00,-13.33
labors_non_vt,8.00,5.00,-3.00,-37.50
drivers_used,12.00,12.00,0.00,0.00
labors_assigned,39.00,39.00,0.00,0.00
labors_unassigned,14.00,5.00,-9.00,-64.29
labors_infeasible,2.00,0.00,-2.00,-100.00
labors_infeasible_pct,3.77,0.00,-3.77,-100.00


In [5]:
build_comparison_gantt_figure(pair_s1_speed, "baseline", "S1 — OSRM Speed").show()

In [8]:
build_comparison_service_figure(pair_s1_speed, "baseline", "S1 — OSRM Speed").show()

In [9]:
build_comparison_driver_figure(pair_s1_speed, "baseline", "S1 — OSRM Speed").show()

---
## 3. S2 — OSRM Time Method
Production baseline vs `setup_1/output_time` (`tiempo_previo=0`, OSRM duration endpoint for travel times).

In [ ]:
build_overview_table(pair_s1_time, "baseline", "S2 — OSRM Time")

In [ ]:
build_comparison_gantt_figure(pair_s1_time, "baseline", "S2 — OSRM Time").show()

In [ ]:
build_comparison_service_figure(pair_s1_time, "baseline", "S2 — OSRM Time").show()

In [ ]:
build_comparison_driver_figure(pair_s1_time, "baseline", "S2 — OSRM Time").show()

---
## 4. Cross-Scenario: OSRM Speed vs OSRM Time

Direct comparison of the two algorithm outputs (`S1-Speed` as A, `S2-Time` as B).  
Shows how using OSRM's duration endpoint instead of distance÷speed affects the solution.

In [ ]:
build_overview_table(pair_methods, "S1 — OSRM Speed", "S2 — OSRM Time")

In [ ]:
build_comparison_gantt_figure(pair_methods, "S1 — OSRM Speed", "S2 — OSRM Time").show()

In [ ]:
build_comparison_service_figure(pair_methods, "S1 — OSRM Speed", "S2 — OSRM Time").show()

In [ ]:
build_comparison_driver_figure(pair_methods, "S1 — OSRM Speed", "S2 — OSRM Time").show()

### 4d. Travel-Time Distribution: Speed-based vs OSRM Duration

VT labors are identified by `labor_distance_km > 0` (non-VT labors have 0 km fixed-location distance).

- **Left**: overlaid histograms of `duration_min` for VT labors in each method's solution  
- **Right**: per-labor scatter — points below the diagonal mean OSRM-time produced a *shorter* duration than speed-based

In [ ]:
# VT labors: labor_distance_km > 0 (non-VT have fixed shop location → 0 km)
_df_sp = pd.DataFrame([
    r for r in pair_methods.rows_a
    if (r.get("labor_distance_km") or 0) > 0 and r.get("duration_min")
])
_df_tm = pd.DataFrame([
    r for r in pair_methods.rows_b
    if (r.get("labor_distance_km") or 0) > 0 and r.get("duration_min")
])

_merged = pd.DataFrame()
if not _df_sp.empty and not _df_tm.empty:
    _merged = _df_sp[["labor_id", "duration_min"]].merge(
        _df_tm[["labor_id", "duration_min"]],
        on="labor_id",
        suffixes=("_speed", "_time"),
    )

_diag_max = 0.0
if not _merged.empty:
    _diag_max = max(_merged["duration_min_speed"].max(), _merged["duration_min_time"].max())

fig_tt = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Duration distribution — VT labors",
        "Per-labor: Speed-based vs OSRM-time duration",
    ],
    horizontal_spacing=0.12,
)

# ── Histograms ─────────────────────────────────────────────────────────────
if not _df_sp.empty:
    fig_tt.add_trace(
        go.Histogram(
            x=_df_sp["duration_min"], name="OSRM Speed",
            opacity=0.75, marker_color="rgba(55, 115, 200, 0.8)", nbinsx=20,
        ),
        row=1, col=1,
    )
if not _df_tm.empty:
    fig_tt.add_trace(
        go.Histogram(
            x=_df_tm["duration_min"], name="OSRM Time",
            opacity=0.75, marker_color="rgba(250, 100, 50, 0.8)", nbinsx=20,
        ),
        row=1, col=1,
    )

# ── Scatter with diagonal reference ────────────────────────────────────────
if not _merged.empty:
    fig_tt.add_trace(
        go.Scatter(
            x=_merged["duration_min_speed"],
            y=_merged["duration_min_time"],
            mode="markers",
            marker=dict(size=7, color="rgba(80, 80, 80, 0.65)", line=dict(width=0.5, color="white")),
            text=_merged["labor_id"].astype(str),
            hovertemplate=(
                "Labor %{text}<br>"
                "Speed: %{x:.1f} min<br>"
                "Time:  %{y:.1f} min"
                "<extra></extra>"
            ),
            name="Labors",
            showlegend=False,
        ),
        row=1, col=2,
    )
    fig_tt.add_trace(
        go.Scatter(
            x=[0, _diag_max], y=[0, _diag_max],
            mode="lines",
            line=dict(color="rgba(150, 150, 150, 0.6)", dash="dash", width=1),
            name="Speed = Time",
        ),
        row=1, col=2,
    )

fig_tt.update_layout(
    height=460,
    barmode="overlay",
    legend=dict(orientation="h", y=1.10),
    title_text="Travel Time: OSRM Speed-based vs OSRM Duration",
)
fig_tt.update_xaxes(title_text="Duration (min)",             row=1, col=1)
fig_tt.update_yaxes(title_text="Count",                      row=1, col=1)
fig_tt.update_xaxes(title_text="Speed-based duration (min)", row=1, col=2)
fig_tt.update_yaxes(title_text="OSRM-time duration (min)",   row=1, col=2)
fig_tt.show()

# ── Summary stats ──────────────────────────────────────────────────────────
if not _merged.empty:
    _diff = _merged["duration_min_time"] - _merged["duration_min_speed"]
    print(f"Duration difference (OSRM Time − Speed) across {len(_merged)} VT labors:")
    print(f"  Mean   : {_diff.mean():+.1f} min")
    print(f"  Median : {_diff.median():+.1f} min")
    print(f"  Std    : {_diff.std():.1f} min")
    print(f"  Range  : [{_diff.min():+.1f},  {_diff.max():+.1f}] min")
    print(f"  OSRM-time faster on {(_diff < 0).sum()} / {len(_diff)} labors")